# NexusAI Core-01 — RUN4 SINGLE UPLOAD & PREFLIGHT NOTEBOOK (v2)

**One ZIP. Validate. Stage. Verify checkpoint. Inspect resume. NO TRAINING.**

Upload only `NEXUSAI_CORE01_RUN4_SINGLE_UPLOAD_BUNDLE.zip` (~81 MB) — or place it
on Drive first. The Run3 checkpoint stays on Drive and is verified read-only.

Cells 1–9 are validation-only and safe to rerun. Cell 10 is a manual gate.

## Cell 1 — Environment (read-only)

In [ ]:
import sys, os, platform, torch
print("python:", sys.version.split()[0])
print("torch:", torch.__version__, "| cuda:", torch.cuda.is_available())
if torch.cuda.is_available(): print("gpu:", torch.cuda.get_device_name(0))

## Cell 2 — Google Drive Mount + storage check

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import shutil
base='/content/drive/MyDrive'
print('MyDrive ok:', os.path.isdir(base))
print('NexusAI dir:', os.path.isdir(base+'/NexusAI'))
t,u,f = shutil.disk_usage('/content/drive/MyDrive')
print(f'Drive free: {f/1e9:.1f} GB')

## Cell 3 — SINGLE UPLOAD / DRIVE DISCOVERY + VALIDATION

Drive-first: looks for `MyDrive/NexusAI/NEXUSAI_CORE01_RUN4_SINGLE_UPLOAD_BUNDLE.zip`.
Fallback: ONE upload widget for the single ZIP (current-session only — if it
errors, put the ZIP on Drive and rerun; the Drive path needs no widget).

In [ ]:
# =============================================================================
# NEXUSAI CORE-01 RUN4 — SINGLE UPLOAD CELL v3.0 (Colab, self-contained)
# ONE ZIP in -> verify everything -> stage to Drive -> report. NO TRAINING.
# Manifest-driven extraction (exact archive paths), streaming SHA-256
# (385MB corpus never fully loaded), payload hashes are the trust anchor.
# Idempotent: identical files reused, conflicts stop safely.
# CELL_VERSION printed first — stale cells are detectable.
# =============================================================================
import os, sys, json, glob, hashlib, zipfile, shutil, tempfile, re
from datetime import datetime

CELL_VERSION = "v3.1"
print(f"[CELL_VERSION] {CELL_VERSION} — Run4 single-upload validator")

# Trusted release identity. The NOTEBOOK-embedded copy has these injected
# with the real values; the copy inside the ZIP keeps the placeholder
# (self-reference is impossible — a file cannot contain its own archive hash).
# Empty/placeholder => outer fingerprint recorded but NOT gated; every
# payload is still verified against the trusted hashes below.
RELEASE_SHA256 = "8b2129649c840e1b7009ea57babacd0e3831185852a070cd4391fbcdda3849ef"
RELEASE_SIZE = 173914048

REPORT = {"ts": datetime.now().isoformat(), "checks": {}, "files": {},
          "warnings": [], "critical_failures": [],
          "TRAINING_STARTED": False, "TRAINING_AUTHORIZED": False,
          "PREFLIGHT_ONLY": True, "status": "BLOCKED"}

def note(k, v): REPORT["checks"][k] = v
def critical(k, m):
    note(k, "FAIL — " + m); REPORT["critical_failures"].append(f"{k}: {m}")
def warn(k, m):
    note(k, "WARN — " + m); REPORT["warnings"].append(f"{k}: {m}")
def sha256_file(p, chunk=1 << 22):
    h = hashlib.sha256()
    with open(p, "rb") as f:
        for c in iter(lambda: f.read(chunk), b""):
            h.update(c)
    return h.hexdigest()
def compute_status():
    if REPORT["critical_failures"]: return "BLOCKED"
    if REPORT["warnings"]: return "READY WITH WARNINGS"
    return "READY FOR USER REVIEW"

BUNDLE_NAME = "NEXUSAI_CORE01_RUN4_SINGLE_UPLOAD_BUNDLE.zip"
BUNDLE_STEM = "NEXUSAI_CORE01_RUN4_SINGLE_UPLOAD_BUNDLE"
BUNDLE_RE = re.compile(
    r"^" + re.escape(BUNDLE_STEM) + r"(?: \(\d+\))?\.zip$", re.IGNORECASE)

# Trusted payload identities (hardcoded here = trust anchor; the notebook
# cell is the authoritative copy, NOT the copy inside the ZIP).
EXPECTED = {
    "COLAB_RUN4_PREFLIGHT_CELL.py":
        ("62fec3a7265bcf843d262d0241b92c95b623faf721fb7c2f1b981c75b2ea08ef",
         30708, "preflight",
         "/content/nexusai_run4_preflight/COLAB_RUN4_PREFLIGHT_CELL.py"),
    "NEXUSAI_CORE01_RUN4_BUNDLE.zip":
        ("778ded61720651951e1dc24ce02e3c36aa520c187498e9657d826af3a050110e",
         300734, "trainzip",
         "DRIVE:NexusAI/NEXUSAI_CORE01_RUN4_BUNDLE.zip"),
    "corpus_train.jsonl":
        ("5f8572edd816ac875a7f52915bc6326635d7a483e0ca3f33b70fffdda7e28376",
         385217021, "corpus",
         "DRIVE:NexusAI/data/run4/corpus_train.jsonl"),
    "nexusai_run4_resume_smoke_evidence.json":
        ("0cb7588d7f2e7afe2e5250dd9381f3c37e1940428cb060477929bb1338523f32",
         2932, "evidence",
         "DRIVE:NexusAI/nexusai_run4_resume_smoke_evidence.json"),
    "checkpoint_final.pt":
        ("ad515cadea433d1782e9f1d5f8806d850ee519b86b42b4b9db11c3b3f4bb7830",
         106777501, "checkpoint",
         "DRIVE:NexusAI/checkpoints/core01_english_run3/checkpoint_final.pt"),
}
REQUIRED_KINDS = {"preflight", "trainzip", "corpus", "evidence", "checkpoint"}

# ------------------------------------------------------------- Drive -------
DRIVE_OK = False
try:
    from google.colab import drive
    drive.mount("/content/drive")
    DRIVE_OK = os.path.isdir("/content/drive/MyDrive")
    note("drive_mount", "ok" if DRIVE_OK else "mounted but MyDrive missing")
except Exception as e:
    note("drive_mount", f"FAILED: {e}")
if not DRIVE_OK:
    critical("drive_mount", "Drive unavailable — cannot stage/report safely")
BASE = "/content/drive/MyDrive/NexusAI" if DRIVE_OK else None
REPORT_DIR = os.path.join(BASE, "reports/run4") if BASE else None
if REPORT_DIR:
    try:
        os.makedirs(REPORT_DIR, exist_ok=True)
        _probe = os.path.join(REPORT_DIR, ".write_probe")
        with open(_probe, "w") as _f:
            _f.write("ok")
        os.remove(_probe)
    except Exception as e:
        critical("report_dir", f"cannot create/write: {e}"); REPORT_DIR = None

# ------------------------------------------------- 1. obtain the bundle ----
zip_path = None
if DRIVE_OK:
    for cand in (os.path.join(BASE, BUNDLE_NAME),
                 os.path.join("/content/drive/MyDrive", BUNDLE_NAME),
                 os.path.join("/content", BUNDLE_NAME)):
        if os.path.exists(cand):
            zip_path = cand; break
    if zip_path:
        note("bundle_source", f"drive:{zip_path}")
if zip_path is None:
    try:
        from google.colab import files
        note("bundle_source", "upload_widget")
        print("=" * 60)
        print("UPLOAD WIDGET: select ONLY the ZIP file")
        print(f"  -> {BUNDLE_NAME}")
        print("  -> Colab may show '...BUNDLE (1).zip' — that is OK")
        print("  -> DO NOT select .py / .ipynb / .json files")
        print("=" * 60)
        ups = files.upload()
        nonzip = [n for n in ups if not n.lower().endswith(".zip")]
        if nonzip:
            critical("bundle_upload",
                     f"non-ZIP file(s) uploaded: {nonzip} — "
                     f"select {BUNDLE_NAME} instead")
        hits = [os.path.join("/content", n) for n in ups if BUNDLE_RE.match(n)]
        if hits:
            zip_path = hits[0]
        else:
            anyz = [os.path.join("/content", n) for n in ups
                    if n.lower().endswith(".zip")]
            if anyz:
                critical("bundle_name",
                         f"unrelated zip uploaded: {anyz[0]} — expected "
                         f"{BUNDLE_NAME} (or Colab ' (N)' duplicate suffix)")
    except Exception as e:
        critical("bundle_upload", f"upload failed: {e}")

if not zip_path or not os.path.exists(zip_path):
    critical("bundle_zip", f"{BUNDLE_NAME} not found via Drive or upload")

# ---- outer fingerprint (informational; payload hashes are the trust anchor)
outer_ok = bool(zip_path and os.path.exists(zip_path))
if outer_ok:
    try:
        osz = os.path.getsize(zip_path); osh = sha256_file(zip_path)
        release_pinned = (RELEASE_SHA256 not in ("", "8b2129649c840e1b7009ea57babacd0e3831185852a070cd4391fbcdda3849ef")
                          and RELEASE_SIZE > 0)
        release_match = (osh == RELEASE_SHA256 and osz == RELEASE_SIZE) \
            if release_pinned else None
        REPORT["files"]["outer_bundle"] = {
            "path": zip_path, "size": osz, "sha256": osh,
            "release_pinned": release_pinned,
            "release_match": release_match}
        if release_pinned:
            note("outer_bundle",
                 "PASS" if release_match else "FAIL — not the trusted release")
            if not release_match:
                critical("outer_bundle",
                         f"size={osz} sha={osh[:16]}… != pinned release "
                         f"{RELEASE_SIZE}B/{RELEASE_SHA256[:16]}…")
                outer_ok = False
        else:
            note("outer_bundle",
                 f"recorded {osz}B sha={osh[:16]}… (release sha not pinned "
                 "in this copy; payload hashes remain authoritative)")
        canon = os.path.join("/content", BUNDLE_NAME)
        if os.path.abspath(zip_path) != os.path.abspath(canon):
            try:
                if os.path.exists(canon):
                    if sha256_file(canon) == osh:
                        os.remove(zip_path); zip_path = canon
                    else:
                        critical("bundle_canonical",
                                 f"{canon} exists with different content")
                        outer_ok = False
                else:
                    shutil.copy2(zip_path, canon); zip_path = canon
                note("bundle_canonical", f"ok -> {zip_path}")
            except Exception as e:
                critical("bundle_canonical", f"rename/copy failed: {e}")
                outer_ok = False
    except Exception as e:
        critical("outer_bundle", f"read error: {e}"); outer_ok = False

# ------------------------------------- 2. manifest-driven verify+extract ---
def norm_arc(n):
    return n.replace("\\", "/").lstrip("/")

def is_unsafe(n):
    p = norm_arc(n).split("/")
    return (n.startswith(("/", "\\")) or ".." in p
            or bool(re.match(r"^[A-Za-z]:", n)) or "" in p)

def stream_extract(zf, arc, dst):
    """Stream one member to dst while hashing; returns (sha, size)."""
    h = hashlib.sha256(); size = 0
    with zf.open(arc) as src, open(dst, "wb") as out:
        for c in iter(lambda: src.read(1 << 22), b""):
            h.update(c); out.write(c); size += len(c)
    return h.hexdigest(), size

stage_dir = None
if outer_ok:
    try:
        z = zipfile.ZipFile(zip_path)
        bad = z.testzip()
        names = z.namelist()
        trav = [n for n in names if is_unsafe(n)]
        # reject symlink entries (unix mode S_IFLNK in external_attr)
        links = [n for n in names
                 if (z.getinfo(n).external_attr >> 16) & 0o170000 == 0o120000]
        # reject duplicate archive paths
        dups = [n for n in set(names) if names.count(n) > 1]
        if bad is not None:
            critical("bundle_zip", f"corrupt member: {bad}")
        elif trav:
            critical("bundle_zip", f"unsafe paths: {trav}")
        elif links:
            critical("bundle_zip", f"symlink entries: {links}")
        elif dups:
            critical("bundle_zip", f"duplicate archive paths: {dups}")
        else:
            note("bundle_zip", f"ok — {len(names)} entries, testzip clean")
            # --- manifest is the structural authority ---
            man = None
            for cand in ("manifest.json", "MANIFEST.json"):
                if cand in names:
                    try:
                        man = json.loads(z.read(cand))
                    except Exception as e:
                        critical("manifest", f"unparseable: {e}")
                    break
            if man is None:
                critical("manifest", "manifest.json missing")
            mfiles = {}
            if isinstance(man, dict):
                for f in man.get("files", []):
                    p = norm_arc(f.get("path", ""))
                    if p:
                        mfiles[p] = f
            # resolve EXPECTED basenames to exactly one manifest path each
            resolved = {}
            for base_name, (exp_sha, exp_size, kind, stage) in EXPECTED.items():
                hits = [p for p in mfiles
                        if p.rsplit("/", 1)[-1] == base_name]
                if len(hits) != 1:
                    critical("payload_" + kind,
                             f"{base_name}: {len(hits)} manifest matches "
                             f"(need exactly 1): {hits}")
                    continue
                resolved[kind] = hits[0]
            # verify + extract every manifest-declared file (streaming)
            stage_dir = tempfile.mkdtemp(prefix="run4_stage_")
            allok = True
            for arc, f in mfiles.items():
                if arc not in names:
                    critical("manifest_entry", f"declared but absent: {arc}")
                    allok = False; continue
                safe_local = arc.replace("/", "_")
                dst = os.path.join(stage_dir, safe_local)
                try:
                    h, sz = stream_extract(z, arc, dst)
                except Exception as e:
                    critical("extract_" + arc, f"failed: {e}"); allok = False
                    continue
                exp_sha = f.get("sha256"); exp_size = f.get("size")
                ok = (exp_sha is None or h == exp_sha) and \
                     (exp_size is None or sz == exp_size)
                REPORT["files"][arc] = {"size": sz, "sha256": h,
                                        "manifest_sha": exp_sha, "match": ok}
                if not ok:
                    critical("payload_hash_" + arc,
                             f"hash/size mismatch got {h[:16]}…/{sz}")
                    allok = False
            # trusted-identity gate on the 5 required payloads
            for base_name, (exp_sha, exp_size, kind, stage) in EXPECTED.items():
                arc = resolved.get(kind)
                if arc is None:
                    continue  # already critical'd above
                info = REPORT["files"].get(arc, {})
                if not (info.get("sha256") == exp_sha
                        and info.get("size") == exp_size):
                    critical("payload_trusted_" + kind,
                             f"{base_name} != trusted sha/size")
                    allok = False
                else:
                    note("payload_" + kind, f"PASS ({arc})")
            REPORT["checks"]["payloads_verified"] = allok
            # dataset streaming sanity (first 2000 non-empty lines)
            corpus_arc = resolved.get("corpus")
            if corpus_arc and allok:
                try:
                    with z.open(corpus_arc) as cf:
                        cnt = 0; badline = 0
                        for raw in cf:
                            if not raw.strip():
                                continue
                            try:
                                json.loads(raw)
                            except Exception:
                                badline += 1
                            cnt += 1
                            if cnt >= 2000:
                                break
                    note("dataset_stream_sample",
                         f"first {cnt} records parsed, {badline} malformed")
                    if badline:
                        warn("dataset_stream_sample", f"{badline} malformed in sample")
                except Exception as e:
                    warn("dataset_stream_sample", f"sample failed: {e}")
            # -------------------------------- 3. stage to Drive -------------
            if allok and DRIVE_OK:
                for kind, arc in resolved.items():
                    if kind not in REQUIRED_KINDS:
                        continue
                    base_name, (_, _, _, stage) = next(
                        (b, v) for b, v in EXPECTED.items() if v[2] == kind)
                    sp = os.path.join(stage_dir, arc.replace("/", "_"))
                    target = (stage if not stage.startswith("DRIVE:")
                              else os.path.join("/content/drive", stage[6:]))
                    try:
                        os.makedirs(os.path.dirname(target), exist_ok=True)
                        if os.path.exists(target):
                            if sha256_file(target) == sha256_file(sp):
                                note("stage_" + kind,
                                     "already present & identical — skip")
                                continue
                            critical("stage_" + kind,
                                     f"target exists with DIFFERENT content: {target}")
                            continue
                        shutil.copy2(sp, target)
                        if sha256_file(target) != sha256_file(sp):
                            critical("stage_verify_" + kind,
                                     "staged copy hash mismatch")
                        else:
                            note("stage_" + kind, f"ok -> {target}")
                    except Exception as e:
                        critical("stage_" + kind, f"copy failed: {e}")
            # -------------------------------- 4. resume-state spot check ---
            if allok and DRIVE_OK:
                ck = os.path.join("/content/drive/MyDrive/NexusAI",
                                  "checkpoints/core01_english_run3/checkpoint_final.pt")
                if os.path.exists(ck):
                    try:
                        import torch
                        d = torch.load(ck, map_location="cpu", weights_only=False)
                        ok = (d.get("epoch") == 3 and d.get("step") == 1341
                              and "optimizer_state_dict" in d
                              and "scheduler_state_dict" in d
                              and "rng_state" in d)
                        note("resume_state",
                             "FULL" if ok else "INCOMPLETE — see fields")
                        REPORT["files"]["resume_state"] = {
                            "epoch": d.get("epoch"), "step": d.get("step"),
                            "status": d.get("model_status"),
                            "has_optimizer": "optimizer_state_dict" in d,
                            "has_scheduler": "scheduler_state_dict" in d,
                            "has_scaler": d.get("grad_scaler_state_dict") is not None,
                            "rng_keys": sorted((d.get("rng_state") or {}).keys()),
                        }
                        if not ok:
                            critical("resume_state", "missing required fields")
                        del d
                    except Exception as e:
                        critical("resume_state", f"load failed: {e}")
                else:
                    critical("resume_state", "checkpoint missing after staging")
        z.close()
    except Exception as e:
        critical("bundle_zip", f"cannot process: {e}")

REPORT["status"] = compute_status()

# ------------------------------------------------------------- report ------
report_paths = {}
if REPORT_DIR:
    stamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    def render():
        j = json.dumps(REPORT, indent=1, ensure_ascii=False)
        return {"json": j, "txt": j,
                "md": "# RUN4 SINGLE-UPLOAD REPORT v3\n\n```json\n" + j + "\n```\n"}
    def wp(ext):
        b = os.path.join(REPORT_DIR, f"RUN4_SINGLE_UPLOAD_{stamp}.{ext}")
        p, i = b, 1
        while os.path.exists(p):
            p = b.replace(f".{ext}", f"_{i}.{ext}"); i += 1
        return p
    for ext in ("json", "txt", "md"):
        p = wp(ext)
        try:
            open(p, "w", encoding="utf-8").write(render()[ext])
            report_paths[ext] = p
        except Exception as e:
            critical("report_write", f"{ext}: {e}")
    REPORT["status"] = compute_status()

print("\n================= SINGLE UPLOAD RESULT v3 =================")
print(json.dumps(REPORT["checks"], indent=1, ensure_ascii=False))
print("critical:", REPORT["critical_failures"])
print("warnings:", REPORT["warnings"])
print("reports:", report_paths)
print("STATUS:", REPORT["status"])
print("TRAINING_STARTED:", REPORT["TRAINING_STARTED"],
      "| TRAINING_AUTHORIZED:", REPORT["TRAINING_AUTHORIZED"])
if REPORT["status"] == "READY FOR USER REVIEW":
    print("\nNext: run the preflight cell, then review reports/run4/.")


## Cell 6b — Bounded GPU Smoke Test (v1.0, smoke only — NOT training)

Verifies GPU/CUDA, FP16/GradScaler, forward+backward, strict checkpoint load,
and temp save/reload. One forward, one backward, at most one throwaway-optimizer
step. Writes nothing to Drive checkpoints. Run AFTER Cell 6, BEFORE Cell 8.


In [ ]:
# =============================================================================
# NEXUSAI CORE-01 RUN4 — BOUNDED GPU SMOKE TEST CELL (REVIEW DRAFT v1.0)
# PURPOSE: verify GPU runtime path ONLY. NOT training. NOT a production checkpoint.
# Safe to insert as a NEW cell AFTER Cell 6 (upload/staging) in
#   docs/notebooks/NEXUSAI_CORE01_RUN4_SINGLE_UPLOAD.ipynb
# Hard limits: batch<=2, seq_len<=64, ONE forward, ONE backward, ONE optional
# scaler step on a THROWAWAY optimizer. Temp dir only. Never touches Drive
# checkpoints or the Run3/production artifacts.
# =============================================================================
import os, sys, json, glob, hashlib, zipfile, shutil, tempfile, time, re
import torch

CELL_VERSION = "RUN4_GPU_SMOKE_V1_0"
print("[CELL_VERSION]", CELL_VERSION)

RESULTS = []
def check(name, ok, detail="", status=None):
    st = status or ("PASS" if ok else "FAIL")
    RESULTS.append({"check": name, "status": st, "detail": str(detail)[:300]})
    print(f"[{st}] {name}: {detail}")
    return ok

def sha256_stream(path, limit=None):
    h = hashlib.sha256()
    n = 0
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1 << 22), b""):
            h.update(chunk); n += len(chunk)
            if limit and n >= limit: break
    return h.hexdigest()

# ---------------------------------------------------------------- 1. GPU gate
if not torch.cuda.is_available():
    print("[BLOCKED] CUDA unavailable — GPU smoke test cannot run.")
    print(json.dumps({"status": "BLOCKED", "reason": "no_cuda"}, indent=2))
    raise SystemExit

dev = "cuda"
gpu_name = torch.cuda.get_device_name(0)
gpu_mem_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
check("gpu_available", True, f"{gpu_name} | {gpu_mem_gb:.1f} GB")
if gpu_mem_gb < 4.0:
    print("[BLOCKED] GPU memory < 4GB — insufficient for smoke test.")
    raise SystemExit

fp16_ok = torch.cuda.is_bf16_supported() or True  # fp16 autocast supported on all modern CUDA
try:
    x = torch.ones(2, 2, device=dev, dtype=torch.float16)
    (x @ x).sum().item()
    fp16_ok = True
except Exception as e:
    fp16_ok = False
check("fp16_tensor_support", fp16_ok,
      "supported" if fp16_ok else "NOT_SUPPORTED", "PASS" if fp16_ok else "NOT_SUPPORTED")

# ------------------------------------------------- 2. locate verified assets
DRIVE_BASE = "/content/drive/MyDrive/NexusAI"
INNER_ZIP_SHA = "f91411e1ac2793c4c8c1a5b5b90c74c4409382a2933b3862e3bcb4d9bd82b4a5"
INNER_ZIP_SIZE = 299476
CKPT_SHA = "ad515cadea433d1782e9f1d5f8806d850ee519b86b42b4b9db11c3b3f4bb7830"
CKPT_SIZE = 106777501

candidates = []
candidates += glob.glob(os.path.join(DRIVE_BASE, "**", "NEXUSAI_CORE01_RUN4_BUNDLE*.zip"), recursive=True)
candidates += glob.glob("/content/NEXUSAI_CORE01_RUN4_BUNDLE*.zip")
inner_zip = None
for c in candidates:
    if os.path.getsize(c) == INNER_ZIP_SIZE and sha256_stream(c) == INNER_ZIP_SHA:
        inner_zip = c; break
if not check("inner_bundle_verified", inner_zip is not None,
             inner_zip or "no verified inner bundle found on Drive/content"):
    raise SystemExit

ckpt_candidates = glob.glob(os.path.join(DRIVE_BASE, "**", "checkpoint_final.pt"), recursive=True)
ckpt_path = None
for c in ckpt_candidates:
    if "core01_english_run3" in c.replace("\\", "/") and \
       os.path.getsize(c) == CKPT_SIZE and sha256_stream(c) == CKPT_SHA:
        ckpt_path = c; break
if not check("run3_checkpoint_verified", ckpt_path is not None,
             ckpt_path or "no hash-verified Run3 checkpoint found"):
    raise SystemExit

# ------------------------------------------------------- 3. temp extraction
WORK = tempfile.mkdtemp(prefix="nexusai_gpu_smoke_")
print("[WORKDIR]", WORK)
try:
    seen_paths = set()
    with zipfile.ZipFile(inner_zip) as z:
        bad = z.testzip()
        if bad: raise RuntimeError(f"corrupt zip entry: {bad}")
        # extract ONLY code+tokenizer assets needed for smoke (small files)
        need_prefixes = ("packages/ai-core/core01/", "packages/ai-core/models/",
                         "packages/ai-core/data/experimental/phase9_2/tokenizers/")
        for info in z.infolist():
            raw = info.filename
            n = raw.replace("\\", "/")
            if n.startswith("/") or ".." in n.split("/") or "" in n.split("/"):
                raise RuntimeError(f"unsafe path in bundle: {raw}")
            if "\\" in raw or re.match(r"^[A-Za-z]:", raw):
                raise RuntimeError(f"unsafe path in bundle: {raw}")
            mode = (info.external_attr >> 16) & 0o170000
            if mode == 0o120000:
                raise RuntimeError(f"symlink entry rejected: {raw}")
            norm = n.lstrip("/")
            if norm in seen_paths:
                raise RuntimeError(f"duplicate archive path: {raw}")
            seen_paths.add(norm)
            if n.startswith(need_prefixes):
                z.extract(info, WORK)
    ai_core = os.path.join(WORK, "packages", "ai-core")
    sys.path.insert(0, ai_core)
    sys.path.insert(0, os.path.join(ai_core, "models"))

    # sentencepiece dependency
    try:
        import sentencepiece  # noqa
    except ImportError:
        import subprocess
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "sentencepiece"])

    from core01.transformer import NexusAICore01
    from core01.tokenizer_interface import NexusAITokenizer
    from nexusai_core_01.config.model_config import (
        NexusAICore01Config, ModelStatus)
    check("imports", True, "core01.transformer + tokenizer_interface + config")

    # ------------------------------------------------- 4. load checkpoint
    ck = torch.load(ckpt_path, map_location="cpu", weights_only=False)
    # weights_only=False is REQUIRED: our own checkpoint holds numpy RNG +
    # optimizer state. Trust boundary: sha256 verified above before this load.
    cfg_dict = dict(ck["config"])
    if isinstance(cfg_dict.get("model_status"), str):
        cfg_dict["model_status"] = ModelStatus[cfg_dict["model_status"]]
    cfg_fields = {f for f in NexusAICore01Config.__dataclass_fields__}
    cfg = NexusAICore01Config(**{k: v for k, v in cfg_dict.items() if k in cfg_fields})
    check("config_built", cfg.vocab_size == 8000 and cfg.d_model == 256,
          f"vocab={cfg.vocab_size} d_model={cfg.d_model} layers={cfg.n_layers}")

    model = NexusAICore01(cfg)
    model.load_state_dict(ck["model_state_dict"], strict=True)
    n_params = sum(p.numel() for p in model.parameters())
    check("strict_state_dict_load", True,
          f"{n_params:,} params loaded strict=True (expected ~8,842,560)")

    tok_path = os.path.join(ai_core, "data", "experimental", "phase9_2",
                            "tokenizers", "bpe_8k", "sp_bpe_8k.model")
    tok = NexusAITokenizer(tok_path, cfg)
    ids_en = tok.encode("Hello, how are you?")
    ids_bn = tok.encode("আপনি কেমন আছেন?")
    check("tokenizer_encode", len(ids_en) > 2 and len(ids_bn) > 2,
          f"en_ids={len(ids_en)} bn_ids={len(ids_bn)} vocab={tok.get_vocab_size()}")

    model.to(dev).train()
    torch.manual_seed(42)

    # ------------------------------------------- 5. bounded forward pass
    BATCH, SEQ = 2, 32   # hard limits — far below training batch (8x~500)
    vocab = cfg.vocab_size
    input_ids = torch.randint(2, vocab, (BATCH, SEQ), device=dev)
    pad_id = cfg.pad_id
    input_ids[:, -4:] = pad_id  # exercise padding path
    attn = (input_ids != pad_id).long()

    t0 = time.time()
    with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=fp16_ok):
        logits = model(input_ids, attention_mask=attn)
    fwd_ms = (time.time() - t0) * 1000
    ok_shape = tuple(logits.shape) == (BATCH, SEQ, vocab)
    ok_finite = torch.isfinite(logits.float()).all().item()
    check("forward_pass", ok_shape and ok_finite,
          f"logits{tuple(logits.shape)} finite={ok_finite} {fwd_ms:.0f}ms")
    if not (ok_shape and ok_finite):
        raise RuntimeError("forward pass failed")

    # ------------------------------------- 6. backward + grad validation
    # Manual causal-LM loss: predict token t+1 from logits at t.
    labels = input_ids[:, 1:]
    pred = logits[:, :-1, :].float()
    loss = torch.nn.functional.cross_entropy(
        pred.reshape(-1, vocab), labels.reshape(-1), ignore_index=pad_id)
    check("loss_finite", torch.isfinite(loss).item(), f"loss={loss.item():.4f}")

    loss.backward()
    grads = [p.grad for p in model.parameters() if p.grad is not None]
    finite_grads = all(torch.isfinite(g.float()).all().item() for g in grads)
    n_with_grad = len(grads)
    total_params = sum(1 for _ in model.parameters())
    check("backward_grads", finite_grads and n_with_grad == total_params,
          f"{n_with_grad}/{total_params} tensors have finite grads")
    if not (finite_grads and n_with_grad == total_params):
        raise RuntimeError("backward/gradient validation failed")

    # grad clipping smoke (matches config grad_clip=1.0)
    torch.nn.utils.clip_grad_norm_(model.parameters(), cfg.grad_clip)
    check("grad_clip", True, f"max_norm<= {cfg.grad_clip}")

    # -------------------------------------- 7. FP16/GradScaler verification
    if fp16_ok:
        try:
            from torch.amp import GradScaler
            scaler = GradScaler(device="cuda")
            opt = torch.optim.AdamW(model.parameters(), lr=1e-5)
            model.zero_grad(set_to_none=True)
            with torch.autocast(device_type="cuda", dtype=torch.float16):
                logits2 = model(input_ids, attention_mask=attn)
            loss2 = torch.nn.functional.cross_entropy(
                logits2[:, :-1, :].float().reshape(-1, vocab),
                labels.reshape(-1), ignore_index=pad_id)
            scaler.scale(loss2).backward()
            scaler.unscale_(opt)
            gn = torch.nn.utils.clip_grad_norm_(model.parameters(), cfg.grad_clip)
            # ONE bounded step on a THROWAWAY optimizer — checkpoint untouched
            scaler.step(opt); scaler.update()
            model.zero_grad(set_to_none=True)
            check("fp16_gradscaler", torch.isfinite(loss2).item(),
                  f"scaled backward+step OK, grad_norm={gn:.3f}")
        except Exception as e:
            check("fp16_gradscaler", False, f"FAIL: {e}")
    else:
        check("fp16_gradscaler", None, "GPU/torch lacks fp16 — NOT_SUPPORTED",
              status="NOT_SUPPORTED")

    # ------------------------------------- 8. save/reload verification
    sd_path = os.path.join(WORK, "smoke_state_dict.pt")
    torch.save(model.state_dict(), sd_path)
    model2 = NexusAICore01(cfg)
    model2.load_state_dict(torch.load(sd_path, map_location="cpu",
                                    weights_only=True), strict=True)
    same = all(torch.equal(model2.state_dict()[k].cpu(),
                           model.state_dict()[k].cpu())
               for k in model.state_dict())
    check("save_reload_state_dict", same and os.path.getsize(sd_path) > 1_000_000,
          f"state_dict roundtrip identical, {os.path.getsize(sd_path):,} bytes")
    if not same:
        raise RuntimeError("save/reload mismatch")

    peak_mb = torch.cuda.max_memory_allocated() / 1e6
    check("gpu_memory_peak", peak_mb < 6000, f"peak={peak_mb:.0f} MB (<6000 limit)")

finally:
    # ------------------------------------------------------- 9. cleanup
    del model, logits, loss
    try:
        import gc; gc.collect()
    except Exception:
        pass
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.reset_peak_memory_stats()
    shutil.rmtree(WORK, ignore_errors=True)
    print("[CLEANUP] temp dir removed, GPU cache cleared")

# ------------------------------------------------------------------ verdict
fails = [r for r in RESULTS if r["status"] == "FAIL"]
nots = [r for r in RESULTS if r["status"] == "NOT_SUPPORTED"]
print("\n===== GPU SMOKE SUMMARY =====")
for r in RESULTS:
    print(f"  {r['status']:14} {r['check']}: {r['detail']}")
verdict = "FAIL" if fails else ("PASS_WITH_NOT_SUPPORTED" if nots else "PASS")
print(f"\nSMOKE_RESULT: {verdict}")
print("TRAINING_STARTED: False | TRAINING_AUTHORIZED: False | SMOKE_ONLY: True")


## Cell 4 — Run3 checkpoint: exact Drive hash verification (read-only)

In [ ]:
# CELL 5 — Run3 checkpoint: Drive discovery + exact hash comparison (read-only)
import os, hashlib, time
BASE = "/content/drive/MyDrive/NexusAI"
CKPT = os.path.join(BASE, "checkpoints/core01_english_run3/checkpoint_final.pt")
EXP_SIZE = 106777501
EXP_SHA  = "ad515cadea433d1782e9f1d5f8806d850ee519b86b42b4b9db11c3b3f4bb7830"
print("expected path:", CKPT)
if not os.path.exists(CKPT):
    print("DRIVE_CHECKPOINT_MISSING")
else:
    sz = os.path.getsize(CKPT)
    h = hashlib.sha256()
    with open(CKPT, "rb") as f:
        for c in iter(lambda: f.read(1 << 22), b""):
            h.update(c)
    sha = h.hexdigest()
    print("ts:", time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()))
    print("size:", sz, "match:", sz == EXP_SIZE)
    print("sha256:", sha)
    print("sha match:", sha == EXP_SHA)
    print("readable: True")
    print("RESULT:", "DRIVE_CHECKPOINT_VERIFIED" if (sz == EXP_SIZE and sha == EXP_SHA)
          else "DRIVE_CHECKPOINT_MISMATCH")

## Cell 5 — Complete resume-state inspection (read-only)

In [ ]:
# CELL 5 — Complete resume-state inspection (READ-ONLY, no training)
# weights_only=False is required: this is OUR OWN self-produced checkpoint
# containing numpy RNG state + optimizer state (not in safe-globals allowlist).
# Trust boundary: file hash was verified against the trusted constant in Cell 4.
import os, torch, json
CKPT = "/content/drive/MyDrive/NexusAI/checkpoints/core01_english_run3/checkpoint_final.pt"
ck = torch.load(CKPT, map_location="cpu", weights_only=False)
req = ["model_state_dict","optimizer_state_dict","scheduler_state_dict",
       "rng_state","epoch","step","config","model_status"]
missing = [k for k in req if k not in ck]
print("type:", type(ck).__name__)
for k in ("epoch","step","model_status","train_loss","val_loss","timestamp","tokenizer_path"):
    print(f"{k}: {ck.get(k)}")
print("model_state_dict tensors:", len(ck.get("model_state_dict", {})))
print("optimizer keys:", list(ck.get("optimizer_state_dict", {}).keys()))
sd = ck.get("scheduler_state_dict") or {}
print("scheduler last_epoch:", sd.get("last_epoch"), "| _step_count:", sd.get("_step_count"))
print("scaler state:", ck.get("grad_scaler_state_dict"))
rng = ck.get("rng_state") or {}
print("rng keys:", sorted(rng.keys()))
cfg = ck.get("config", {})
print("arch:", {k: cfg.get(k) for k in ("d_model","n_layers","n_heads","d_ff","max_len","vocab_size","model_name")})
print("cuda rng tensors present:", "torch_cuda" in rng)
print("MISSING REQUIRED FIELDS:", missing if missing else "NONE")
print("RESUME_STATE:", "FULL_RESUME_VERIFIED" if not missing else "MISSING_STATE")
del ck

## Cell 6 — Architecture & tokenizer compatibility

In [ ]:
# CELL 7 — Architecture & tokenizer compatibility (constants-based, no repo needed)
# Compares checkpoint config to the verified Run3/Run4 architecture constants.
import os, torch
CKPT = "/content/drive/MyDrive/NexusAI/checkpoints/core01_english_run3/checkpoint_final.pt"
ck = torch.load(CKPT, map_location="cpu", weights_only=False)
cfg = ck.get("config", {})
EXPECTED = {"d_model":256,"n_layers":6,"n_heads":8,"d_ff":1024,"max_len":512,"vocab_size":8000}
bad = {k:(cfg.get(k),v) for k,v in EXPECTED.items() if cfg.get(k)!=v}
print("arch mismatches:", bad if bad else "NONE")
print("tokenizer_path:", ck.get("tokenizer_path"))
print("ARCH_CHECK:", "PASS" if not bad else "FAIL")
del ck

## Cell 7 — PREFLIGHT (validation-only)

Full asset verification + report to `MyDrive/NexusAI/reports/run4/`.

In [ ]:
# =============================================================================
# NEXUSAI CORE-01 RUN4 — FINAL COLAB PREFLIGHT CELL v5 (single cell, no training)
# Read-only verification. FAIL-CLOSED. 3-stage verified report persistence.
# =============================================================================
import os, sys, json, glob, hashlib, zipfile, re, ast
from datetime import datetime

REPORT = {"ts": datetime.now().isoformat(), "checks": {}, "files": {},
          "warnings": [], "missing": [], "critical_failures": [],
          "TRAINING_STARTED": False, "TRAINING_AUTHORIZED": False,
          "PREFLIGHT_ONLY": True, "status": "BLOCKED",
          "checker_version": "v5"}

def note(k, v): REPORT["checks"][k] = v
def critical(k, msg):
    note(k, "FAIL — " + msg); REPORT["critical_failures"].append(f"{k}: {msg}")
def warn(k, msg):
    note(k, "WARN — " + msg); REPORT["warnings"].append(f"{k}: {msg}")

def sha256_file(path, chunk=1 << 22):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for c in iter(lambda: f.read(chunk), b""):
            h.update(c)
    return h.hexdigest()

def all_matches(patterns, roots):
    out = []
    for pat in patterns:
        for r in roots:
            if r:
                out.extend(sorted(glob.glob(os.path.join(r, pat))))
    seen, res = set(), []
    for p in out:
        if p not in seen:
            seen.add(p); res.append(p)
    return res

# ---------------------------------------------------------------- Drive ------
DRIVE_OK = False
try:
    from google.colab import drive
    drive.mount("/content/drive")
    DRIVE_OK = os.path.isdir("/content/drive/MyDrive")
    note("drive_mount", "ok" if DRIVE_OK else "mounted but MyDrive missing")
except Exception as e:
    note("drive_mount", f"FAILED: {e}")

BASE = "/content/drive/MyDrive/NexusAI" if DRIVE_OK else None
DRIVE_DATA = os.path.join(BASE, "data/run4") if BASE else None
REPORT_DIR = os.path.join(BASE, "reports/run4") if BASE else None
if REPORT_DIR:
    try:
        os.makedirs(REPORT_DIR, exist_ok=True)
    except Exception as e:
        critical("report_dir", f"cannot create: {e}"); REPORT_DIR = None
else:
    critical("drive_mount", "Drive unavailable — cannot verify/report safely")

# ------------------------------------------------------- 1. Run3 ckpt ------
TRUSTED_CKPT_SHA = "ad515cadea433d1782e9f1d5f8806d850ee519b86b42b4b9db11c3b3f4bb7830"
TRUSTED_CKPT_SIZE = 106777501
if BASE:
    ckpt = os.path.join(BASE, "checkpoints/core01_english_run3/checkpoint_final.pt")
    if os.path.exists(ckpt):
        try:
            size = os.path.getsize(ckpt); sha = sha256_file(ckpt)
            ok = (size == TRUSTED_CKPT_SIZE and sha == TRUSTED_CKPT_SHA)
            REPORT["files"]["run3_checkpoint"] = {"path": ckpt, "size": size,
                                                  "sha256": sha, "match": ok}
            note("run3_checkpoint", "PASS" if ok else "FAIL")
            if not ok: critical("run3_checkpoint", "size/hash MISMATCH")
        except Exception as e:
            critical("run3_checkpoint", f"read error: {e}")
    else:
        REPORT["files"]["run3_checkpoint"] = {"path": ckpt, "missing": True}
        critical("run3_checkpoint", "missing on Drive")

# ------------------------------------------------------- 2. Bundle ZIP -----
TRUSTED_ZIP_SHA = "f91411e1ac2793c4c8c1a5b5b90c74c4409382a2933b3862e3bcb4d9bd82b4a5"
TRUSTED_ZIP_SIZE = 299476
roots = tuple(r for r in ("/content", DRIVE_DATA, BASE) if r)
zip_candidates = all_matches(
    ["NEXUSAI_CORE01_RUN4_BUNDLE*.zip", "NEXUSAI_CORE01_RUN4_BUNDLE*.ZIP"], roots)
zip_path, cand_info = None, []
for p in zip_candidates:
    try:
        sz = os.path.getsize(p); sh = sha256_file(p)
        cand_info.append({"path": p, "size": sz, "sha256": sh})
        if sz == TRUSTED_ZIP_SIZE and sh == TRUSTED_ZIP_SHA:
            zip_path = p
    except Exception as e:
        cand_info.append({"path": p, "error": str(e)})
REPORT["files"]["bundle_zip_candidates"] = cand_info
if zip_candidates and zip_path is None:
    critical("bundle_zip_hash", "no candidate matches trusted sha256/size")
elif not zip_candidates:
    critical("bundle_zip", "not found")
elif len([c for c in cand_info if c.get("sha256") == TRUSTED_ZIP_SHA]) > 1:
    warn("bundle_zip", "multiple identical trusted copies — deterministic pick")

if zip_path:
    REPORT["files"]["bundle_zip"] = {"path": zip_path, "size": TRUSTED_ZIP_SIZE,
                                     "sha256": TRUSTED_ZIP_SHA, "selected_by_hash": True}
    note("bundle_zip_hash", "PASS")
    try:
        z = zipfile.ZipFile(zip_path); bad = z.testzip(); names = z.namelist()
        def _unsafe(n):
            parts = n.replace("\\", "/").split("/")
            return (n.startswith(("/", "\\")) or ".." in parts
                    or bool(re.match(r"^[A-Za-z]:", n)))
        traversal = [n for n in names if _unsafe(n)]
        cell_name = "docs/notebooks/COLAB_RUN4_TRAIN_CELL.py"
        required = [cell_name,
                    "packages/ai-core/core01/pretraining_driver.py",
                    "packages/ai-core/core01/transformer.py",
                    "packages/ai-core/core01/training.py",
                    "packages/ai-core/core01/dataset.py",
                    "packages/ai-core/core01/tokenizer_interface.py",
                    "packages/ai-core/models/nexusai_core_01/config/model_config.py"]
        missing_entries = [r for r in required if r not in names]
        tok_ok = all(any(n.endswith(s) for n in names) for s in
                     ("sp_bpe_8k.model", "sp_bpe_8k.vocab"))
        TRUSTED_CELL_SHA = ("fda564afe3703cd459321d28b363aefe1f26ebf8652e5a2b"
                            "2842cffa2f0580c6")
        cell_src_b = z.read(cell_name) if cell_name in names else b""
        try:
            cell_src = cell_src_b.decode("utf-8", errors="strict") if cell_src_b else ""
        except UnicodeDecodeError:
            cell_src = ""
        cell_sha = hashlib.sha256(cell_src_b).hexdigest()
        cell_sha_ok = (cell_sha == TRUSTED_CELL_SHA)

        # ---- resume wiring: structural AST data-flow ----
        wiring = {"ck_load_line": None, "start_step_line": None,
                  "start_epoch_line": None, "del_ck_line": None,
                  "rep_dict_line": None, "rep_keys": {},
                  "rep_consumers": [], "cmd_line": None,
                  "resume_arg": False, "resume_src_used": False,
                  "subprocess_call_line": None, "evidence": []}
        resume_ok = False
        try:
            _t = ast.parse(cell_src)
            rep_dict_node = None
            cmd_list_node = None
            # locate assignments & calls
            for _n in ast.walk(_t):
                if isinstance(_n, ast.Assign):
                    for _tg in _n.targets:
                        if not isinstance(_tg, ast.Name):
                            continue
                        if _tg.id == "ck" and isinstance(_n.value, ast.Call):
                            wiring["ck_load_line"] = _n.lineno
                        if _tg.id in ("START_STEP", "START_EPOCH") \
                                and "ck" in ast.dump(_n.value):
                            wiring["start_step_line" if _tg.id == "START_STEP"
                                   else "start_epoch_line"] = _n.lineno
                        if _tg.id == "rep" and isinstance(_n.value, ast.Dict):
                            rep_dict_node = _n
                            wiring["rep_dict_line"] = _n.lineno
                        if _tg.id == "cmd" and isinstance(_n.value, ast.List):
                            cmd_list_node = _n.value
                            wiring["cmd_line"] = _n.lineno
                elif isinstance(_n, ast.Delete):
                    for _tg in _n.targets:
                        if isinstance(_tg, ast.Name) and _tg.id == "ck":
                            wiring["del_ck_line"] = _n.lineno
                elif isinstance(_n, ast.Call):
                    # consumer: subprocess.run(cmd, ...) or json.dump(rep, ...)
                    fn = _n.func
                    fname = ""
                    if isinstance(fn, ast.Attribute):
                        fname = fn.attr
                    elif isinstance(fn, ast.Name):
                        fname = fn.id
                    for a_ in _n.args:
                        if isinstance(a_, ast.Name) and a_.id == "cmd":
                            wiring["subprocess_call_line"] = _n.lineno
                        if isinstance(a_, ast.Name) and a_.id == "rep":
                            wiring["rep_consumers"].append(
                                {"call": fname, "line": _n.lineno})
            if rep_dict_node is not None:
                for k_, v_ in zip(rep_dict_node.value.keys,
                                  rep_dict_node.value.values):
                    if isinstance(k_, ast.Constant) and isinstance(v_, ast.Name):
                        wiring["rep_keys"][k_.value] = v_.id
            if cmd_list_node is not None:
                vals = []
                for el in cmd_list_node.elts:
                    if isinstance(el, ast.Constant):
                        vals.append(el.value)
                    elif isinstance(el, ast.Name):
                        vals.append(el.id)
                wiring["resume_arg"] = "--resume" in vals
                wiring["resume_src_used"] = "RESUME_SRC" in vals
            # data-flow verdict
            ok_flow = (wiring["ck_load_line"] and wiring["start_step_line"]
                       and wiring["start_epoch_line"] and wiring["del_ck_line"]
                       and wiring["start_step_line"] < wiring["del_ck_line"]
                       and wiring["start_epoch_line"] < wiring["del_ck_line"]
                       and wiring["rep_dict_line"]
                       and wiring["rep_keys"].get("start_step") == "START_STEP"
                       and wiring["rep_keys"].get("start_epoch") == "START_EPOCH"
                       and wiring["rep_consumers"]
                       and wiring["cmd_line"] and wiring["subprocess_call_line"]
                       and wiring["resume_arg"] and wiring["resume_src_used"])
            resume_ok = bool(ok_flow)
            wiring["evidence"].append(
                "START_STEP/START_EPOCH are report metadata; actual resume "
                "flows via RESUME_SRC -> cmd '--resume' -> driver "
                "load_checkpoint (verified statically).")
            wiring["driver_evidence"] = (
                "Driver path verified statically AND by bounded smoke test "
                "(2026-11-11): --resume -> engine.load_checkpoint -> "
                "CheckpointManager restores model(strict)/optimizer/"
                "scheduler/rng + current_epoch/current_step "
                "(training.py:469-486); loop continues from restored epoch. "
                "Real Run3 ckpt: epoch=3 step=1341 restored; exactly ONE "
                "optimizer step executed; temp ckpt saved+reloaded. See "
                "external_verification for validated machine-readable "
                "evidence. Full training-loop resume NOT executed.")
        except Exception as e:
            wiring["error"] = str(e)
        REPORT["files"]["bundle_zip"]["resume_wiring"] = wiring
        REPORT["files"]["bundle_zip"].update({
            "entries": len(names), "testzip": bad,
            "traversal_entries": traversal, "missing_entries": missing_entries,
            "cell_sha256": cell_sha, "cell_sha_match": cell_sha_ok,
            "resume_order_ok": bool(resume_ok), "tokenizer_files": tok_ok})
        if bad is None and not traversal and not missing_entries:
            note("zip_integrity", "PASS")
        else:
            critical("zip_integrity", f"testzip={bad} trav={len(traversal)} miss={missing_entries}")
        if cell_sha_ok: note("cell_sha_match", "PASS")
        else: critical("cell_sha_match", f"got {cell_sha[:16]}")
        if resume_ok: note("resume_wiring", "PASS")
        else: critical("resume_wiring", "AST wiring incomplete")
        if tok_ok: note("tokenizer_in_bundle", "PASS")
        else: critical("tokenizer_in_bundle", "missing sp_bpe_8k")
        z.close()
    except Exception as e:
        critical("zip_integrity", f"cannot read zip: {e}")

# ------------------------------------------------------- 3. Corpus JSONL ----
# Authoritative corpus SHA-256 provenance (repo evidence):
#   eval_reports/nexusai_gate_b_backup_manifest.json ->
#     orig=packages/ai-core/data/phase2/run4_corpus/corpus_train.jsonl
#     size=385217021 sha256=5f8572ed... (computed directly on the raw file)
#   docs/architecture/audits/NEXUSAI_PHASE2_BLOCKER_FIX_EXECUTION_REPORT.md
#     records the same hash as the rebuilt corpus SHA-256.
TRUSTED_CORPUS_SHA = "5f8572edd816ac875a7f52915bc6326635d7a483e0ca3f33b70fffdda7e28376"
TRUSTED_CORPUS_SIZE = 385217021
CORPUS_PROVENANCE = ("eval_reports/nexusai_gate_b_backup_manifest.json "
                     "(orig=corpus_train.jsonl, hash computed on raw file) + "
                     "NEXUSAI_PHASE2_BLOCKER_FIX_EXECUTION_REPORT.md")
roots2 = tuple(r for r in ("/content", DRIVE_DATA, "/content/drive/MyDrive") if r)
corpus_candidates = all_matches(["corpus_train.jsonl"], roots2)
corpus, cand_info = None, []
for p in corpus_candidates:
    try:
        sz = os.path.getsize(p); sh = sha256_file(p)
        cand_info.append({"path": p, "size": sz, "sha256": sh})
        if sz == TRUSTED_CORPUS_SIZE and sh == TRUSTED_CORPUS_SHA:
            corpus = p
    except Exception as e:
        cand_info.append({"path": p, "error": str(e)})
REPORT["files"]["corpus_candidates"] = cand_info
REPORT["files"]["corpus_provenance"] = CORPUS_PROVENANCE
if corpus_candidates and corpus is None:
    critical("corpus_identity",
             "CORPUS_IDENTITY_UNVERIFIED — no candidate matches trusted "
             "sha256+size")
elif corpus is None:
    critical("corpus", "corpus_train.jsonl not found")
elif corpus:
    note("corpus_identity", "CORPUS_IDENTITY_VERIFIED")

if corpus:
    try:
        size = os.path.getsize(corpus)
        total_phys = n = ok = empty = mal = dup = blank = non_dict = enc_err = 0
        seen = set()
        with open(corpus, "rb") as fb:
            for raw in fb:
                total_phys += 1
                if not raw.strip():
                    blank += 1; continue
                n += 1
                try:
                    line = raw.decode("utf-8")
                except UnicodeDecodeError:
                    enc_err += 1; continue
                try:
                    d = json.loads(line)
                except Exception:
                    mal += 1; continue
                if not isinstance(d, dict):
                    non_dict += 1; continue
                ok += 1
                t = d.get("text", "")
                if not isinstance(t, str) or not t.strip():
                    empty += 1
                else:
                    h = hashlib.md5(t.encode("utf-8")).hexdigest()
                    if h in seen: dup += 1
                    seen.add(h)
        rep = {"path": corpus, "size": size,
               "total_physical_lines": total_phys,
               "total_non_empty_lines": n, "records": n, "parsed": ok,
               "valid_records": ok, "malformed_json": mal,
               "encoding_errors": enc_err, "encoding_error_lines": enc_err,
               "blank_lines": blank, "empty_text": empty,
               "non_dict_records": non_dict, "duplicate_texts": dup,
               "unique_texts": len(seen), "strict_utf8": enc_err == 0,
               "dup_policy": "counted, non-fatal", "lines": n,
               "sha256": TRUSTED_CORPUS_SHA}
        REPORT["files"]["corpus"] = rep
        if (size == TRUSTED_CORPUS_SIZE and mal == 0 and empty == 0
                and non_dict == 0 and enc_err == 0 and n == 216193):
            note("corpus", "PASS — strict UTF-8 + structure valid")
        else:
            critical("corpus", f"size={size} records={n} mal={mal} "
                               f"enc_err={enc_err} empty={empty} non_dict={non_dict}")
    except Exception as e:
        critical("corpus", f"validation error: {e}")

# ------------------------------------------------- 4. Optional manifest ----
mani_candidates = all_matches(["corpus_manifest.json"], roots2)
if mani_candidates:
    mani = mani_candidates[0]
    try:
        m = json.load(open(mani, encoding="utf-8"))
        info = {"path": mani, "type": type(m).__name__,
                "note": "informational only — not authoritative provenance"}
        if isinstance(m, dict):
            info["keys"] = list(m)[:15]
            c = REPORT["files"].get("corpus", {})
            mism = []
            for mk, ck_ in (("num_docs","records"),("docs","records"),
                            ("document_count","records"),("records","records"),
                            ("lines","records"),("num_lines","records"),
                            ("size_bytes","size"),("bytes","size")):
                if mk in m and ck_ in c:
                    try:
                        if int(m[mk]) != int(c[ck_]):
                            mism.append(f"{mk}={m[mk]} vs actual {c[ck_]}")
                    except Exception:
                        mism.append(f"{mk}={m[mk]!r} not comparable")
            if mism:
                info["consistency_mismatches"] = mism
                warn("corpus_manifest", "inconsistent: " + "; ".join(mism))
        REPORT["files"]["corpus_manifest"] = info
        note("corpus_manifest", "present (optional, informational)")
    except Exception as e:
        warn("corpus_manifest", f"invalid/unreadable: {e}")

# ---------------------------------------------------------- verdict ---------
def compute_status():
    if REPORT["critical_failures"]:
        return "BLOCKED"
    if REPORT["warnings"]:
        return "READY WITH WARNINGS"
    return "READY FOR USER REVIEW"
REPORT["status"] = compute_status()

# ------------- external verification evidence (validated, fail-closed) ------
# The bounded Run3 resume smoke test was executed locally on 2026-11-11.
# Its machine-readable evidence is persisted at
#   eval_reports/nexusai_run4_resume_smoke_evidence.json  (repo)
# For Colab, upload that file to the Drive NexusAI folder. This cell does NOT
# trust hardcoded claims: it locates the evidence file, parses it, and
# validates every critical field against the trusted constants. Missing,
# malformed, or inconsistent evidence => critical => BLOCKED.
EVIDENCE_NAME = "nexusai_run4_resume_smoke_evidence.json"
EXPECTED_EVIDENCE = {
    "restored_epoch": 3, "restored_step": 1341, "post_step": 1342,
    "device": "cpu", "precision": "fp32",
    "smoke_status": "PRETRAINING_IN_PROGRESS",
    "temp_ckpt_sha": ("f7236ce35580ed33308b8c589c872e2cded32f2253"
                      "cbaed24db30a2d90b35e88"),
    "original_sha": TRUSTED_CKPT_SHA,
    "min_tests": 15,
}
ev_roots = tuple(r for r in ("/content", DRIVE_DATA, "/content/drive/MyDrive",
                             BASE) if r)
ev_candidates = all_matches([EVIDENCE_NAME], ev_roots)
# Pin the evidence file's trusted identity. Provenance: the JSON was written
# programmatically by the executed smoke-test script (not hand-edited);
# its sha256/size below were computed from those exact bytes on 2026-11-11.
# A sha256 pin detects modification; it does NOT prove the event occurred —
# authenticity rests on the preserved artifacts + documented generation.
EXPECTED_EVIDENCE_SHA = ("0cb7588d7f2e7afe2e5250dd9381f3c37e1940428cb060477"
                         "929bb1338523f32")
EXPECTED_EVIDENCE_SIZE = 2932
ev_info = []
for _p in ev_candidates:
    try:
        ev_info.append({"path": _p, "size": os.path.getsize(_p),
                        "sha256": sha256_file(_p)})
    except Exception as _e:
        ev_info.append({"path": _p, "error": str(_e)})
REPORT["files"]["smoke_evidence_candidates"] = ev_info
trusted_ev = [c for c in ev_info
              if c.get("sha256") == EXPECTED_EVIDENCE_SHA
              and c.get("size") == EXPECTED_EVIDENCE_SIZE]
distinct_ev = {c.get("sha256") for c in ev_info if "sha256" in c}
REPORT["external_verification"] = {
    "evidence_file": EVIDENCE_NAME,
    "evidence_candidates": ev_candidates,
    "corpus_identity_status": "PENDING_VALIDATION",
    "resume_flow_status": "UNVERIFIED",
    "checkpoint_compatibility_status": "UNVERIFIED",
    "CUDA_FP16_EXECUTION_STATUS": "NOT_TESTED",
    "SCHEDULER_BEHAVIOR": "PARTIALLY_VERIFIED",
    "training_execution_authorized": False,
    "full_run4_training_executed": False,
}
if not ev_candidates:
    critical("smoke_evidence",
             f"{EVIDENCE_NAME} not found — upload it to the Drive NexusAI "
             "folder; resume smoke evidence cannot be validated")
elif len(distinct_ev) > 1:
    critical("smoke_evidence",
             "ambiguous evidence candidates with DIFFERENT hashes: "
             + json.dumps(ev_info))
elif not trusted_ev:
    critical("smoke_evidence",
             "evidence file hash/size does not match trusted identity "
             f"(expected sha256={EXPECTED_EVIDENCE_SHA[:16]}…, "
             f"size={EXPECTED_EVIDENCE_SIZE})")
else:
    ev_path = trusted_ev[0]["path"]
    if len(trusted_ev) > 1:
        warn("smoke_evidence",
             "multiple identical trusted copies — deterministic pick")
    try:
        ev = json.load(open(ev_path, encoding="utf-8"))
        tests = ev.get("tests", [])
        ed = ev.get("evidence", {})
        fails = []
        if not isinstance(tests, list) or len(tests) < EXPECTED_EVIDENCE["min_tests"]:
            fails.append(f"tests missing/short: {len(tests) if isinstance(tests,list) else '?'}")
        elif any(t.get("result") != "PASS" for t in tests):
            fails.append("one or more smoke tests did not PASS")
        checks = {
            "restored_epoch": ed.get("restored_epoch"),
            "restored_step": ed.get("restored_step"),
            "post_step": ed.get("post_step"),
            "device": ed.get("device"),
            "precision": ed.get("precision"),
            "smoke_status": ed.get("smoke_status"),
            "temp_ckpt_sha": ed.get("temp_ckpt_sha"),
            "original_sha_pre": ed.get("original_sha_pre"),
            "original_sha_post": ed.get("original_sha_post"),
        }
        for k, exp in EXPECTED_EVIDENCE.items():
            if k in checks and checks[k] != exp:
                fails.append(f"{k}={checks[k]!r} expected {exp!r}")
        if ed.get("original_sha_pre") != TRUSTED_CKPT_SHA or \
           ed.get("original_sha_post") != TRUSTED_CKPT_SHA:
            fails.append("original ckpt sha pre/post != trusted sha")
        if fails:
            critical("smoke_evidence",
                     f"evidence inconsistent at {ev_path}: " + "; ".join(fails))
        else:
            REPORT["external_verification"].update({
                "corpus_identity_status": "CORPUS_IDENTITY_VERIFIED",
                "resume_flow_status": "RESUME_FLOW_VERIFIED_BOUNDED_SMOKE",
                "checkpoint_compatibility_status": "COMPATIBLE",
                "evidence_path": ev_path,
                "evidence_sha256": EXPECTED_EVIDENCE_SHA,
                "evidence_size": EXPECTED_EVIDENCE_SIZE,
                "evidence_integrity": "PINNED_HASH_MATCH",
                "tests_passed": f"{len(tests)}/{len(tests)}",
                "restored": {"epoch": ed["restored_epoch"],
                             "step": ed["restored_step"],
                             "post_step": ed["post_step"]},
                "temp_ckpt": {"size": ed.get("temp_ckpt_size"),
                              "sha256": ed.get("temp_ckpt_sha")},
                "original_ckpt_integrity": "UNCHANGED (sha256 pre==post)",
                "device": ed["device"], "precision": ed["precision"],
                "limitations": (
                    "CPU/FP32 only — CUDA/FP16 GradScaler path verified in "
                    "source, not exercised. Scheduler lr-lambda rebuilt with "
                    "Run4 step budget; last_epoch restored. Bounded smoke "
                    "test proves resume mechanics, NOT full Run4 training."),
            })
            note("smoke_evidence", "PASS — validated against trusted constants")
    except Exception as e:
        critical("smoke_evidence", f"unreadable/malformed evidence: {e}")

# --------------------------- 5. three-stage verified report persistence -----
report_paths = {}
if REPORT_DIR:
    stamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    def render():
        j = json.dumps(REPORT, indent=1, ensure_ascii=False)
        return {"json": j, "txt": j,
                "md": "# RUN4 FINAL PREFLIGHT REPORT\n\n```json\n" + j + "\n```\n"}
    def unique_path(ext, tag=""):
        # Stage tag in filename prevents a stale earlier-stage report from
        # being mistaken for the final verdict (e.g. stageA "initial" vs
        # stageB/C final files are now distinguishable on disk).
        base_p = os.path.join(
            REPORT_DIR,
            f"RUN4_PREFLIGHT_{stamp}{('_' + tag) if tag else ''}.{ext}")
        p, i = base_p, 1
        while os.path.exists(p):
            p = base_p.replace(f".{ext}", f"_{i}.{ext}"); i += 1
        return p
    def write_stage(tag):
        """Write all 3 reports; verify existence/size/sha/parse; return dict."""
        paths, res = {}, {"ok": True, "files": {}, "errors": []}
        for ext in ("json", "txt", "md"):
            p = unique_path(ext, tag)
            try:
                with open(p, "w", encoding="utf-8") as f:
                    f.write(render()[ext])
                ok = os.path.exists(p) and os.path.getsize(p) > 0
                sha = sha256_file(p) if ok else None
                paths[ext] = p
                res["files"][ext] = {"path": p,
                                     "size": os.path.getsize(p) if ok else 0,
                                     "sha256": sha}
                if not ok:
                    res["ok"] = False; res["errors"].append(f"{ext} empty/missing")
            except Exception as e:
                res["ok"] = False; res["errors"].append(f"{ext}: {e}")
        # cross-consistency: JSON parses + status field matches
        try:
            pj = json.load(open(paths["json"], encoding="utf-8"))
            if pj.get("status") != REPORT["status"]:
                res["ok"] = False
                res["errors"].append("json status mismatch")
            for ext in ("txt", "md"):
                if REPORT["status"] not in open(paths[ext], encoding="utf-8").read():
                    res["ok"] = False
                    res["errors"].append(f"{ext} missing status")
        except Exception as e:
            res["ok"] = False; res["errors"].append(f"reparse: {e}")
        res["stage"] = tag
        return res, paths
    # Stage A: initial write
    try:
        stageA, report_paths = write_stage("initial")
    except Exception as e:
        stageA = {"ok": False, "errors": [str(e)]}
        report_paths = {}
    REPORT["report_write_stageA"] = stageA
    # Stage B: final write (content now includes stageA result + verdict)
    REPORT["status"] = compute_status()
    REPORT["_status_at_B"] = REPORT["status"]
    try:
        stageB, report_paths = write_stage("final")
    except Exception as e:
        stageB = {"ok": False, "errors": [str(e)]}
        report_paths = {}
    REPORT["report_write"] = stageB
    if not stageB.get("ok"):
        critical("report_write", "; ".join(stageB.get("errors", ["failed"])))
    REPORT["status"] = compute_status()
    # Stage C correction: if stageB failed or the verdict changed after it,
    # persist the FINAL status once more so on-disk reports cannot contradict
    # the in-memory verdict. Self-hash boundary: these files contain stageA/B
    # results + final status; their own sha is recorded in the response only.
    if (not stageB.get("ok")) or REPORT["status"] != REPORT.get("_status_at_B"):
        try:
            stageC, report_paths = write_stage("final_correction")
        except Exception as e:
            stageC = {"ok": False, "errors": [str(e)]}
        REPORT["report_write_final_correction"] = stageC
        if not stageC.get("ok"):
            critical("report_write_final",
                     "; ".join(stageC.get("errors", ["failed"])))
        REPORT["status"] = compute_status()
    # Last-resort marker: if ALL structured report writes failed, still try a
    # minimal BLOCKED marker so Drive does not only contain stale stage-A files.
    if REPORT["status"] == "BLOCKED" and not report_paths:
        try:
            mp = os.path.join(
                REPORT_DIR, f"RUN4_PREFLIGHT_BLOCKED_{stamp}.txt")
            with open(mp, "w", encoding="utf-8") as _f:
                _f.write(f"RUN4 PREFLIGHT BLOCKED at {REPORT['ts']}\n"
                         f"critical_failures={REPORT['critical_failures']}\n")
            REPORT["blocked_marker"] = mp
        except Exception:
            REPORT["blocked_marker"] = "FAILED — see console output"
    REPORT.pop("_status_at_B", None)
    # Stage C: consistency note — reports contain verdict + stageA/B results;
    # final on-disk files are stageB outputs (self-hash boundary documented).
    REPORT["report_write"]["self_reference_note"] = (
        "stageB files contain the verdict and stageA result; their own "
        "sha256 recorded in stageB response object, not inside the files "
        "(mathematically impossible to embed own hash).")
else:
    critical("report_write", "Drive unavailable — reports not persisted")
    REPORT["status"] = "BLOCKED"

print("\n================= PREFLIGHT =================")
print(json.dumps(REPORT["checks"], indent=1, ensure_ascii=False))
print("critical:", REPORT["critical_failures"])
print("warnings:", REPORT["warnings"])
print("reports:", report_paths)
print("STATUS:", REPORT["status"])
print("TRAINING_STARTED:", REPORT["TRAINING_STARTED"],
      "| TRAINING_AUTHORIZED:", REPORT["TRAINING_AUTHORIZED"],
      "| PREFLIGHT_ONLY:", REPORT["PREFLIGHT_ONLY"])


## Cell 8 — Final summary checklist

After Cells 3–7, confirm:
- [ ] `STATUS:` from upload cell = READY (not BLOCKED)
- [ ] `DRIVE_CHECKPOINT_VERIFIED`
- [ ] resume fields: epoch=3, step=1341, optimizer/scheduler/rng present
- [ ] `ARCH_CHECK: PASS`
- [ ] preflight `STATUS:` = READY FOR USER REVIEW / READY WITH WARNINGS
- [ ] report files exist under `MyDrive/NexusAI/reports/run4/`

In [ ]:
import glob, os
for p in sorted(glob.glob('/content/drive/MyDrive/NexusAI/reports/run4/*'))[-6:]:
    print(os.path.basename(p), os.path.getsize(p), 'bytes')

## Cell 10 — TRAINING AUTHORIZATION GATE (v1.0)

Writes `TRAINING_AUTHORIZATION.json` ONLY if you type the exact phrase
`START NEXUSAI CORE-01 TRAINING` and preflight is not BLOCKED.
The training cell refuses to run without this marker.


In [ ]:
# =============================================================================
# NEXUSAI CORE-01 RUN4 — TRAINING AUTHORIZATION GATE CELL (v1.0)
# Runs AFTER preflight. Writes an authorization marker ONLY if the user types
# the exact phrase. The training cell refuses to run without this marker.
# This cell itself NEVER starts training.
# =============================================================================
import os, json, glob
from datetime import datetime, timezone

REQUIRED_PHRASE = "START NEXUSAI CORE-01 TRAINING"
REPORTS = "/content/drive/MyDrive/NexusAI/reports/run4"
AUTH_FILE = os.path.join(REPORTS, "TRAINING_AUTHORIZATION.json")

print("[GATE_VERSION] RUN4_TRAINING_GATE_V1_0")

if not os.path.isdir(REPORTS):
    print("[BLOCKED] reports/run4 not found — run upload + preflight cells first.")
    raise SystemExit(1)

# latest preflight report must exist and not be BLOCKED
reports = sorted(glob.glob(os.path.join(REPORTS, "RUN4_PREFLIGHT_*.json")))
if not reports:
    print("[BLOCKED] no preflight report found — run Cell 14 (preflight) first.")
    raise SystemExit(1)
latest = reports[-1]
try:
    rep = json.load(open(latest, encoding="utf-8"))
except Exception as e:
    print(f"[BLOCKED] cannot parse preflight report {latest}: {e}")
    raise SystemExit(1)
status = rep.get("status", "UNKNOWN")
print(f"preflight report: {os.path.basename(latest)}")
print(f"preflight status : {status}")
print(f"training flags   : started={rep.get('TRAINING_STARTED')} "
      f"authorized={rep.get('TRAINING_AUTHORIZED')}")
if "BLOCKED" in str(status).upper():
    print("[BLOCKED] preflight status is BLOCKED — fix failures before authorizing.")
    raise SystemExit(1)

phrase = input("Type EXACTLY: START NEXUSAI CORE-01 TRAINING\n> ")
if phrase.strip() != REQUIRED_PHRASE:
    print("[REJECTED] phrase did not match exactly. No authorization written.")
    raise SystemExit(1)

auth = {
    "authorized": True,
    "phrase": REQUIRED_PHRASE,
    "authorized_utc": datetime.now(timezone.utc).isoformat(),
    "preflight_report": os.path.basename(latest),
    "preflight_status": status,
    "note": "Authorizes ONE Run4 training execution via COLAB_RUN4_TRAIN_CELL.py",
}
os.makedirs(REPORTS, exist_ok=True)
tmp = AUTH_FILE + ".tmp"
with open(tmp, "w", encoding="utf-8") as f:
    json.dump(auth, f, indent=1)
os.replace(tmp, AUTH_FILE)
print("[AUTHORIZED] marker written:", AUTH_FILE)
print("Next: run the Run4 training cell MANUALLY. This gate does not start it.")


## Cell 9 — APPROVAL GATE (manual, nothing auto-runs)

Run the training cell (`COLAB_RUN4_TRAIN_CELL.py`) ONLY after explicit approval.
It resumes from the verified Run3 checkpoint into `checkpoints/core01_run4/`.